In [1]:
import pandas as pd
from enum import Enum
from pathlib import Path
import os
import json
from dataclasses import dataclass, field, asdict
from typing import List
import numpy as np
import glob

In [2]:
class UtteranceTypes(Enum): # Not implemented yet
    CONCEDE = "___Concede___"
    RETRACT = "___Retract___"
    WHY = "___Why___"
    QUESTION = "___Question___"
    SINCE = "___Since___"
    CLAIM = "___Claim___"

In [3]:
class ModelName(Enum):
    GPT_4O = "gpt-4o"
    GROK_3 = "grok-3"
    GROK_4 = "grok-4"
    MISTRAL_MEDIUM = "mistral-medium"

In [4]:
@dataclass
class UttTypeCounts:
    claim: int = field(default=0)
    since: int = field(default=0)
    why: int = field(default=0)
    question: int = field(default=0)
    concede: int = field(default=0)
    retract: int = field(default=0)

In [5]:
@dataclass
class IndexData:
    dialogue_list_index: int = field(default=int)
    dialogue_turn_index: int= field(default=int)
    sentence_index: int= field(default=int)

@dataclass
class RetractIndexData:
    dialogue_list_index: int = field(default=int)
    dialogue_turn_index: int= field(default=int)
    sentence_index: int= field(default=int)
    data: list[int] = field(default_factory=lambda: [])

In [6]:
@dataclass
class UttTypeDialogueDataIndices:
    claim: List[IndexData] = field(default_factory=lambda: [])
    since: List[IndexData] = field(default_factory=lambda: [])
    why: List[IndexData] = field(default_factory=lambda: [])
    question: List[IndexData] = field(default_factory=lambda: [])
    concede: List[IndexData] = field(default_factory=lambda: [])
    retract: List[IndexData] = field(default_factory=lambda: [])

In [7]:
@dataclass
class ExampleData:
    speaker: str
    label: str
    target: str
    context: list[str] = field(default=list)
    
    

In [8]:
@dataclass
class ListOfExampleData:
    claim: list[ExampleData] = field(default=list)
    since: list[ExampleData] = field(default=list)
    why: list[ExampleData] = field(default=list)
    question: list[ExampleData] = field(default=list)
    concede: list[ExampleData] = field(default=list)
    retract: list[ExampleData] = field(default=list)

In [9]:
@dataclass
class ModelPromptData:
    gpt_4o: List[ExampleData] = field(default_factory=lambda: [])
    mistral_medium: List[ExampleData] = field(default_factory=lambda: [])
    grok_3: List[ExampleData] = field(default_factory=lambda: [])
    grok_4: List[ExampleData] = field(default_factory=lambda: [])


In [10]:
root_path = Path() / ".." / ".." / "run_experiments" / "model2model" / "data" / "dialogues"

In [11]:
files = os.listdir(root_path)
print(files)

['gpt-4o_base_l__vs__gpt-4o_mas_r.json', 'gpt-4o_base_l__vs__gpt-4o_mas_rag_r.json', 'gpt-4o_base_l__vs__grok-3_base_r.json', 'gpt-4o_base_l__vs__grok-3_mas_r.json', 'gpt-4o_base_l__vs__grok-3_mas_rag_r.json', 'gpt-4o_base_l__vs__grok-4_base_r.json', 'gpt-4o_base_l__vs__grok-4_mas_rag_r.json', 'gpt-4o_base_l__vs__mistral-medium_base_r.json', 'gpt-4o_base_l__vs__mistral-medium_mas_r.json', 'gpt-4o_base_l__vs__mistral-medium_mas_rag_r.json', 'gpt-4o_base_r__vs__gpt-4o_mas_l.json', 'gpt-4o_base_r__vs__gpt-4o_mas_rag_l.json', 'gpt-4o_base_r__vs__grok-3_base_l.json', 'gpt-4o_base_r__vs__grok-3_mas_l.json', 'gpt-4o_base_r__vs__grok-3_mas_rag_l.json', 'gpt-4o_base_r__vs__grok-4_base_l.json', 'gpt-4o_base_r__vs__grok-4_mas_l.json', 'gpt-4o_base_r__vs__grok-4_mas_rag_l.json', 'gpt-4o_base_r__vs__mistral-medium_base_l.json', 'gpt-4o_base_r__vs__mistral-medium_mas_l.json', 'gpt-4o_base_r__vs__mistral-medium_mas_rag_l.json', 'gpt-4o_mas_l__vs__gpt-4o_base_r.json', 'gpt-4o_mas_l__vs__gpt-4o_mas_rag

In [12]:
len(files)

223

We want 10 examples for each type of utterance from all 4 models. So that's 10 x 6 x 4 = 240 examples. Then 4 models will make 5 predictions for each of the 240 examples, thus 240 x 4 x 5 = 4800 completions in total (1200 completions per model)

We can always run another test with 5 repetitions per model so that we have 10 repetitions per model for each sample (cost dependent)

In [13]:
NUM_EXAMPLES = 10

In [14]:
prompt_data = ModelPromptData()
prompt_data

ModelPromptData(gpt_4o=[], mistral_medium=[], grok_3=[], grok_4=[])

In [15]:
dialogue_data = {
    "gpt-4o" : [],
    "mistral-medium" : [],
    "grok-3" : [],
    "grok-4" : []
}

for file in files:
    file_path = root_path / file
    with open(file_path) as f:
        data = json.load(f)

    first_speaker = data["dialogue_history"][0]["speaker"]
    second_speaker = data["dialogue_history"][1]["speaker"]

    for key, _ in dialogue_data.items():
        if (key in first_speaker) or (key in second_speaker):
            dialogue_data[key].append(data["dialogue_history"])

In [16]:
utterance_type_counts = {
    "gpt-4o" : UttTypeCounts(),
    "mistral-medium" : UttTypeCounts(),
    "grok-3" : UttTypeCounts(),
    "grok-4" : UttTypeCounts()
}


utterance_type_indices = {
    "gpt-4o" : UttTypeDialogueDataIndices(),
    "mistral-medium" :  UttTypeDialogueDataIndices(),
    "grok-3" :  UttTypeDialogueDataIndices(),
    "grok-4" :  UttTypeDialogueDataIndices()
}

for key, list_of_dialogues in dialogue_data.items():
    for dialogue_list_index, dialogue in enumerate(list_of_dialogues):
        for dialogue_turn_index, turn in enumerate(dialogue):
            for speaker, _ in utterance_type_counts.items():
                if (speaker in turn["speaker"]) and (key in speaker):
                    for utt_type in UtteranceTypes:
                        for sentence_index, (dialogue_utt_type, sentence) in enumerate(turn["sentences_with_utterance_types"]):
                            if dialogue_utt_type == utt_type.value:
                                attr_name = utt_type.value.lower().replace("_", "")
                                current_count = getattr(utterance_type_counts[speaker], attr_name)
                                setattr(utterance_type_counts[speaker], attr_name, current_count + 1)


                                indices = IndexData(dialogue_list_index, dialogue_turn_index, sentence_index)
                                current_index_list = getattr(utterance_type_indices[speaker], attr_name)
                                current_index_list.append(indices)
                                setattr(utterance_type_indices[speaker], attr_name, current_index_list)

In [17]:
utterance_type_counts

{'gpt-4o': UttTypeCounts(claim=7871, since=1076, why=779, question=418, concede=18, retract=0),
 'mistral-medium': UttTypeCounts(claim=7003, since=847, why=733, question=1270, concede=13, retract=0),
 'grok-3': UttTypeCounts(claim=6224, since=758, why=698, question=555, concede=45, retract=0),
 'grok-4': UttTypeCounts(claim=3829, since=713, why=933, question=1160, concede=10, retract=0)}

Models were seemingly incapable of making retract moves and rarely conceded. So we will have to use of the data from the intermediate generations in our experiments

In [18]:
np.random.choice(utterance_type_indices["gpt-4o"].claim, size=NUM_EXAMPLES, replace=False)

array([IndexData(dialogue_list_index=56, dialogue_turn_index=10, sentence_index=4),
       IndexData(dialogue_list_index=68, dialogue_turn_index=7, sentence_index=1),
       IndexData(dialogue_list_index=18, dialogue_turn_index=2, sentence_index=4),
       IndexData(dialogue_list_index=21, dialogue_turn_index=19, sentence_index=1),
       IndexData(dialogue_list_index=17, dialogue_turn_index=18, sentence_index=0),
       IndexData(dialogue_list_index=97, dialogue_turn_index=15, sentence_index=0),
       IndexData(dialogue_list_index=48, dialogue_turn_index=34, sentence_index=0),
       IndexData(dialogue_list_index=45, dialogue_turn_index=20, sentence_index=0),
       IndexData(dialogue_list_index=19, dialogue_turn_index=20, sentence_index=0),
       IndexData(dialogue_list_index=89, dialogue_turn_index=25, sentence_index=3)],
      dtype=object)

In [19]:
sampled_indices = {
    "gpt-4o" : UttTypeDialogueDataIndices(),
    "mistral-medium" :  UttTypeDialogueDataIndices(),
    "grok-3" :  UttTypeDialogueDataIndices(),
    "grok-4" :  UttTypeDialogueDataIndices()
}


for utt_type in UtteranceTypes:
    if utt_type != UtteranceTypes.RETRACT:
        attr_name = utt_type.value.lower().replace("_", "")
        for model, _ in sampled_indices.items():
            indices = getattr(utterance_type_indices[model], attr_name)
            samples = np.random.choice(indices, size=NUM_EXAMPLES, replace=False)
            setattr(sampled_indices[model], attr_name, samples.tolist())

In [20]:
sampled_indices

{'gpt-4o': UttTypeDialogueDataIndices(claim=[IndexData(dialogue_list_index=52, dialogue_turn_index=15, sentence_index=1), IndexData(dialogue_list_index=53, dialogue_turn_index=19, sentence_index=2), IndexData(dialogue_list_index=107, dialogue_turn_index=19, sentence_index=0), IndexData(dialogue_list_index=59, dialogue_turn_index=38, sentence_index=0), IndexData(dialogue_list_index=58, dialogue_turn_index=16, sentence_index=5), IndexData(dialogue_list_index=31, dialogue_turn_index=2, sentence_index=0), IndexData(dialogue_list_index=5, dialogue_turn_index=26, sentence_index=2), IndexData(dialogue_list_index=26, dialogue_turn_index=30, sentence_index=0), IndexData(dialogue_list_index=64, dialogue_turn_index=1, sentence_index=1), IndexData(dialogue_list_index=21, dialogue_turn_index=9, sentence_index=4)], since=[IndexData(dialogue_list_index=91, dialogue_turn_index=39, sentence_index=0), IndexData(dialogue_list_index=0, dialogue_turn_index=28, sentence_index=2), IndexData(dialogue_list_ind

Now because there are no instances where models make a retract move in the dialogue data, we need to use the intermediate generations to find examples of retract moves.

In [21]:
retract_data_path = Path("..") / ".." / "persuasio" / "persuasio" / "outputs" / "dialogues"
files = list(retract_data_path.rglob("*.json"))

In [22]:
retract_data = {
    "gpt-4o" : [],
    "mistral-medium" : [],
    "grok-3" : [],
    "grok-4" : []
}

retract_dialogue_turn_data = []


dialogue_list_index = 0
for file in files:
    with open(file, encoding="utf-8", errors="ignore") as f:
        data = json.load(f)
    if isinstance(data, dict):
        first_speaker = data["dialogue_history"][0]["speaker"]
        second_speaker = data["dialogue_history"][1]["speaker"]

        for key, _ in retract_data.items():

            if (key in first_speaker):
                if len(data["first_speaker_intermediate_generations"]) != 0:
                    for generation in data["first_speaker_intermediate_generations"]:
                        if UtteranceTypes.RETRACT.value in generation[0].keys():
                            dialogue_turn_index = generation[1]
                            sentence_index = generation[2]
                            candidates = generation[0][UtteranceTypes.RETRACT.value]
                            index_and_data = RetractIndexData(dialogue_list_index, dialogue_turn_index, sentence_index, candidates)
                            retract_data[key].append(index_and_data)

                            retract_dialogue_turn_data.append(data["dialogue_history"])

                            dialogue_list_index += 1 
            elif (key in second_speaker):
                if len(data["second_speaker_intermediate_generations"]) != 0:
                    for generation in data["second_speaker_intermediate_generations"]:
                        if UtteranceTypes.RETRACT.value in generation[0].keys():
                            dialogue_turn_index = generation[1]
                            sentence_index = generation[2]
                            candidates = generation[0][UtteranceTypes.RETRACT.value]
                            index_and_data = RetractIndexData(dialogue_list_index, dialogue_turn_index, sentence_index, candidates)
                            retract_data[key].append(index_and_data)

                            retract_dialogue_turn_data.append(data["dialogue_history"])

                            dialogue_list_index += 1


In [23]:
retract_data

{'gpt-4o': [RetractIndexData(dialogue_list_index=46, dialogue_turn_index=3, sentence_index=3, data=['I retract the claim that fairness and support are better achieved through private sector solutions, as I cannot guarantee the private sector would prioritise the vulnerable over profit.', 'I withdraw the assertion that welfare risks creating dependency, as structured incentives to work can mitigate this concern.', 'I am not committed to the claim that individual initiative alone can address fairness and support without government involvement.']),
  RetractIndexData(dialogue_list_index=47, dialogue_turn_index=5, sentence_index=3, data=['I retract the claim that profit motives encourage businesses to cater to all demographics, including the vulnerable, as their track record often shows otherwise.', 'I withdraw the assertion that relying on the private sector reduces dependency, as it does not guarantee prioritisation of the most vulnerable.', 'I no longer stand by the statement that fairn

In [24]:
assert len(retract_data["gpt-4o"])+ len(retract_data["grok-4"]) + len(retract_data["grok-3"]) + len(retract_data["mistral-medium"]) == len(retract_dialogue_turn_data)

In [25]:
for model, _ in sampled_indices.items():
    attr_name = "retract"
    samples = np.random.choice(retract_data[model], size=NUM_EXAMPLES, replace=False).tolist()
    current_index_list = getattr(sampled_indices[model], attr_name)
    current_index_list.extend(samples)
    setattr(sampled_indices[model], attr_name, current_index_list)


In [26]:
for k, v in sampled_indices['gpt-4o'].__dict__.items():
    print(len(v))

10
10
10
10
10
10


In [27]:
experiments_data = ModelPromptData()

for model, values in sampled_indices.items():
    speaker = model
    current_model_data = getattr(experiments_data, speaker.replace("-", "_"))
    for utt_type, indices_list in values.__dict__.items():
        
        label = utt_type
        
        if utt_type != "retract":
            
            for _samples in indices_list:
                target = dialogue_data[model][_samples.dialogue_list_index][_samples.dialogue_turn_index]["sentences_with_utterance_types"][_samples.sentence_index][1]
                context = []
                if _samples.dialogue_turn_index < 2:
                    for i in range(_samples.dialogue_turn_index - 1, -1, -1):
                        context.append(dialogue_data[model][_samples.dialogue_list_index][i]["sentences_no_utterance_types"])
                else:
                    for i in range(_samples.dialogue_turn_index - 1, _samples.dialogue_turn_index - 3, -1):
                        context.append(dialogue_data[model][_samples.dialogue_list_index][i]["sentences_no_utterance_types"])
                context.reverse()

                current_model_data.append(ExampleData(speaker, label, target, context))

        else:
            for _samples in indices_list:

                target = np.random.choice(_samples.data)
                context = []
                if _samples.dialogue_turn_index < 2:
                    for i in range(_samples.dialogue_turn_index - 1, -1, -1):
                        context.append(retract_dialogue_turn_data[_samples.dialogue_list_index][i]["sentences_no_utterance_types"])
                else:
                    for i in range(_samples.dialogue_turn_index - 1, _samples.dialogue_turn_index - 3, -1):
                        context.append(retract_dialogue_turn_data[_samples.dialogue_list_index][i]["sentences_no_utterance_types"])

                context.reverse()

                current_model_data.append(ExampleData(speaker, label, target, context))

    setattr(experiments_data, speaker.replace("-", "_"), current_model_data)


In [28]:
assert len(experiments_data.mistral_medium) + len(experiments_data.gpt_4o) + len(experiments_data.grok_3) + len(experiments_data.grok_4) == 240

In [29]:
data = []

for k, values in experiments_data.__dict__.items():
    for d in values:
        entry = {
            "dialogue_turn_1" : d.context[0] if len(d.context) > 1 else "",
            "dialogue_turn_2" : d.context[1] if len(d.context) > 1 else d.context[0] if len(d.context) == 1 else "",
            "target" : d.target,
            "label" : d.label,
            "speaker" : d.speaker,

        }

        data.append(entry)

In [30]:
data[:10]

[{'dialogue_turn_1': 'Resource misallocation is more likely in systems with minimal oversight, where there are fewer checks to ensure fairness and proper allocation. Heavy oversight does not create barriers; it ensures that support reaches those who genuinely need it by preventing exploitation. Complex rules are necessary to maintain transparency and accountability, which minimal oversight fails to achieve. Bureaucracy may slow processes, but it is a safeguard against misuse and inequity. Efficiency alone cannot guarantee fairness, as streamlined systems often overlook the needs of the most vulnerable.',
  'dialogue_turn_2': 'Fairness is best achieved through efficiency, as it prioritises timely and equitable access to essential resources. Why should accountability rely on rigid frameworks instead of streamlined systems that promote clarity and efficiency? Heavy oversight creates barriers that delay support and leave those in need struggling to access help. Excessive regulation often p

In [31]:
df = pd.DataFrame.from_dict(data)

In [32]:
len(df)

240

In [33]:
save_dir_path = Path() / ".." / "data" / "experiments" / "utterance_types_testset.csv"

In [34]:
df.to_csv(save_dir_path, index=False, encoding='utf-16', sep="\t")